# WASP XGBoost Kaggle reference experiment

This is a Kaggle-ready, safe replacement for the attached Colab notebook. It recreates its **25-channel / 225-feature, random window-split XGBoost experiment**, but never loads or creates a Python pickle. It writes a native XGBoost JSON model and derived JSON reports to `/kaggle/working`.

> **Important limitation:** this is a reference experiment only. Its random split can put overlapping windows from the same cow and recording in both train and test sets, so its score is optimistic and is not comparable to the repository's nested leave-one-cow-out benchmark. It is not a validation of the project collar, a clinical result, an anomaly detector, or an input to real WASP + MmCows fusion.


## Kaggle input contract

Create a Kaggle Dataset from the **unpacked** `db-cow-walking` folder, then attach it as an input. The notebook discovers one folder containing these four class folders:

- `Walking`
- `Grazing`
- `Resting`
- `Miscellaneous behaviors`

Each source CSV must contain `Time` and the 25 BNO055/MPU9250 columns used by the original notebook. The original friend pickle is deliberately not an input to this notebook.


In [ ]:
from __future__ import annotations

import hashlib
import json
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")
WINDOW_SIZE = 50  # 5 seconds at 10 Hz
STEP_SIZE = 25    # 50% overlap
TEST_SIZE = 0.20

# Kept in the label order used by the attached notebook. These are experiment labels, not platform telemetry codes.
BEHAVIORS = {
    "Walking": 0,
    "Grazing": 1,
    "Resting": 2,
    "Miscellaneous behaviors": 3,
}
LABEL_NAMES = list(BEHAVIORS)

# This deliberately mirrors the supplied notebook's source channels. It is not the repository's six-axis / 112-feature contract.
SENSOR_COLUMNS = [
    "BNO055_ARX", "BNO055_ARY", "BNO055_ARZ",
    "BNO055_AX", "BNO055_AY", "BNO055_AZ",
    "BNO055_GX", "BNO055_GY", "BNO055_GZ",
    "BNO055_MX", "BNO055_MY", "BNO055_MZ",
    "BNO055_Q0", "BNO055_Q1", "BNO055_Q2", "BNO055_Q3",
    "MPU9250_AX", "MPU9250_AY", "MPU9250_AZ",
    "MPU9250_GX", "MPU9250_GY", "MPU9250_GZ",
    "MPU9250_MX", "MPU9250_MY", "MPU9250_MZ",
]
STATISTIC_NAMES = ["mean", "std", "min", "max", "median", "range", "rms", "mean_abs", "energy"]
FEATURE_NAMES = [f"{sensor}__{statistic}" for sensor in SENSOR_COLUMNS for statistic in STATISTIC_NAMES]
assert len(SENSOR_COLUMNS) == 25
assert len(FEATURE_NAMES) == 225

print(json.dumps({
    "seed": SEED,
    "window_samples": WINDOW_SIZE,
    "stride_samples": STEP_SIZE,
    "sensor_columns": len(SENSOR_COLUMNS),
    "feature_count": len(FEATURE_NAMES),
    "split_protocol": "stratified random window split; reference only",
    "xgboost_version": xgboost.__version__,
    "sklearn_version": sklearn.__version__,
}, indent=2))


In [ ]:
# Discover exactly one attached WASP dataset without hard-coding Kaggle's dataset slug.
required_class_dirs = tuple(BEHAVIORS)
candidate_roots = {path.parent.resolve() for path in INPUT_ROOT.rglob("Walking")
                   if path.is_dir() and all((path.parent / class_dir).is_dir() for class_dir in required_class_dirs)}
candidate_roots = sorted(candidate_roots)

if len(candidate_roots) != 1:
    raise FileNotFoundError(
        "Expected exactly one Kaggle input folder containing Walking, Grazing, Resting, and "
        f"Miscellaneous behaviors. Found {len(candidate_roots)} candidate(s): {candidate_roots}"
    )

DATASET_DIR = candidate_roots[0]
CSV_BY_CLASS = {behavior: sorted((DATASET_DIR / behavior).glob("*.csv")) for behavior in BEHAVIORS}
empty_classes = [behavior for behavior, files in CSV_BY_CLASS.items() if not files]
if empty_classes:
    raise FileNotFoundError(f"No CSV files found for: {empty_classes}")

print(json.dumps({
    "dataset_dir": str(DATASET_DIR),
    "csv_files_by_class": {behavior: len(files) for behavior, files in CSV_BY_CLASS.items()},
    "total_csv_files": sum(map(len, CSV_BY_CLASS.values())),
}, indent=2))


In [ ]:
# Build complete 5-second windows. No raw IMU rows are written to output files.
def cow_id_from_path(path: Path) -> str:
    parts = path.stem.split("_")
    if len(parts) < 5:
        raise ValueError(f"Unsupported WASP filename; expected <event>_<label>_<cow>_<date>_<time>.csv: {path.name}")
    return parts[2]

windows: list[np.ndarray] = []
labels: list[int] = []
cow_ids: list[str] = []
source_paths: list[Path] = []
short_recordings = 0

for behavior, label in BEHAVIORS.items():
    for csv_path in CSV_BY_CLASS[behavior]:
        frame = pd.read_csv(csv_path)
        missing = [column for column in ["Time", *SENSOR_COLUMNS] if column not in frame.columns]
        if missing:
            raise ValueError(f"Missing columns in {csv_path.name}: {missing}")
        values = frame[SENSOR_COLUMNS].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
        if not np.isfinite(values).all():
            raise ValueError(f"Non-finite sensor value in {csv_path.name}; correct the input instead of imputing it.")
        if len(values) < WINDOW_SIZE:
            short_recordings += 1
            continue
        for start in range(0, len(values) - WINDOW_SIZE + 1, STEP_SIZE):
            windows.append(values[start:start + WINDOW_SIZE])
            labels.append(label)
            cow_ids.append(cow_id_from_path(csv_path))
            source_paths.append(csv_path)

if not windows:
    raise RuntimeError("No complete 50-sample windows were created.")

X_raw = np.stack(windows).astype(np.float32, copy=False)
y = np.asarray(labels, dtype=np.int64)
cow_ids = np.asarray(cow_ids, dtype=str)
window_summary = {
    "windows": int(len(X_raw)),
    "window_shape": list(X_raw.shape[1:]),
    "short_recordings_excluded": short_recordings,
    "windows_per_class": {name: int((y == label).sum()) for name, label in BEHAVIORS.items()},
    "windows_per_cow": {cow: int(count) for cow, count in sorted(Counter(cow_ids).items())},
}
print(json.dumps(window_summary, indent=2))


In [ ]:
# The nine per-channel statistics preserve the attached notebook's 25 x 9 = 225 feature definition.
def extract_features(window: np.ndarray) -> np.ndarray:
    features: list[float] = []
    for channel in window.T:
        channel = channel.astype(np.float64, copy=False)
        features.extend([
            float(np.mean(channel)), float(np.std(channel)), float(np.min(channel)),
            float(np.max(channel)), float(np.median(channel)),
            float(np.max(channel) - np.min(channel)),
            float(np.sqrt(np.mean(channel ** 2))), float(np.mean(np.abs(channel))),
            float(np.mean(channel ** 2)),
        ])
    return np.asarray(features, dtype=np.float32)

X_features = np.stack([extract_features(window) for window in X_raw])
if X_features.shape != (len(y), len(FEATURE_NAMES)) or not np.isfinite(X_features).all():
    raise RuntimeError(f"Feature contract failure: shape={X_features.shape}, finite={np.isfinite(X_features).all()}")

print(json.dumps({"feature_matrix_shape": list(X_features.shape), "finite": True}, indent=2))


In [ ]:
# This intentionally retains the supplied notebook's random split as a reference.
# Do not use this number as a cow-independent performance claim: overlapping same-recording windows can cross the split.
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
X_train_df = pd.DataFrame(X_train, columns=FEATURE_NAMES)
X_test_df = pd.DataFrame(X_test, columns=FEATURE_NAMES)

xgb_model = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, objective="multi:softprob", num_class=4,
    eval_metric="mlogloss", random_state=SEED, n_jobs=-1, tree_method="hist",
)
xgb_model.fit(X_train_df, y_train)
probabilities = xgb_model.predict_proba(X_test_df)
if probabilities.shape != (len(y_test), len(LABEL_NAMES)) or not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-6):
    raise RuntimeError(f"Invalid probability output: {probabilities.shape}")
y_pred = probabilities.argmax(axis=1)

class_report = classification_report(
    y_test, y_pred, labels=list(range(len(LABEL_NAMES))), target_names=LABEL_NAMES,
    output_dict=True, zero_division=0,
)
metrics = {
    "schema_version": 1,
    "scope": "reference_only",
    "not_valid_for_collar_or_clinical_use": True,
    "split_protocol": "stratified random window split; not grouped by cow or recording",
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "macro_f1": float(f1_score(y_test, y_pred, average="macro", zero_division=0)),
    "test_windows": int(len(y_test)),
    "class_metrics": class_report,
    "confusion_matrix": confusion_matrix(y_test, y_pred, labels=list(range(len(LABEL_NAMES)))).tolist(),
    "mean_max_probability_uncalibrated": float(probabilities.max(axis=1).mean()),
}
print(json.dumps({key: metrics[key] for key in ["accuracy", "macro_f1", "test_windows", "mean_max_probability_uncalibrated"]}, indent=2))
pd.DataFrame(class_report).T.loc[LABEL_NAMES, ["precision", "recall", "f1-score", "support"]]


In [ ]:
# Save only native/auditable and derived artifacts. No .pkl is read or written.
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def source_fingerprint(paths: list[Path]) -> str:
    digest = hashlib.sha256()
    for path in sorted(set(paths)):
        relative = path.relative_to(DATASET_DIR)
        digest.update(f"{relative.as_posix()}:{path.stat().st_size}\n".encode())
    return digest.hexdigest()

MODEL_PATH = OUTPUT_DIR / "cow_behavior_xgboost_reference.json"
MANIFEST_PATH = OUTPUT_DIR / "cow_behavior_xgboost_reference.manifest.json"
METRICS_PATH = OUTPUT_DIR / "cow_behavior_xgboost_reference.metrics.json"

xgb_model.save_model(MODEL_PATH)
model_hash = sha256_file(MODEL_PATH)
selected_parameters = {
    key: (None if isinstance(value, float) and np.isnan(value) else value)
    for key, value in xgb_model.get_params().items()
}
manifest = {
    "schema_version": 1,
    "artifact_scope": "reference_only",
    "eligible_for_project_collar_deployment": False,
    "eligible_for_public_WASP_plus_MmCows_fusion": False,
    "model_format": "native_xgboost_json",
    "model_file": MODEL_PATH.name,
    "model_sha256": model_hash,
    "feature_names": FEATURE_NAMES,
    "feature_sha256": hashlib.sha256(json.dumps(FEATURE_NAMES, separators=(",", ":")).encode()).hexdigest(),
    "sensor_columns": SENSOR_COLUMNS,
    "class_labels": {name: label for name, label in BEHAVIORS.items()},
    "window_settings": {"sample_rate_hz": 10, "window_samples": WINDOW_SIZE, "stride_samples": STEP_SIZE},
    "split_protocol": metrics["split_protocol"],
    "seed": SEED,
    "selected_parameters": selected_parameters,
    "package_versions": {"xgboost": xgboost.__version__, "scikit_learn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__},
    "source_file_metadata_sha256": source_fingerprint(source_paths),
    "legacy_pickle_handling": "not read; not emitted",
}
metrics["model_sha256"] = model_hash

for path, payload in [(MANIFEST_PATH, manifest), (METRICS_PATH, metrics)]:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")

print(json.dumps({
    "status": "complete", "model": str(MODEL_PATH),
    "manifest": str(MANIFEST_PATH), "metrics": str(METRICS_PATH),
    "model_sha256": model_hash,
}, indent=2))


## Interpreting the result

The printed accuracy and macro-F1 measure only this **random window split**. They may look much stronger than the cow-held-out score because adjacent 50-sample windows overlap by 50% and can be divided between train and test. Treat them as a reproducibility reference for your friend's notebook, not as proof that the model works on a new cow or your collar.

The `mean_max_probability_uncalibrated` field is not a calibrated confidence or a health risk. `cow_behavior_xgboost_reference.json` is a native model file; the manifest fixes the exact 25-channel, 225-feature order needed to reproduce predictions.


In [ ]:
# Compact final result for Kaggle's output pane. Download these three files before ending the session.
final_result = {
    "artifact_scope": manifest["artifact_scope"],
    "random_split_accuracy": metrics["accuracy"],
    "random_split_macro_f1": metrics["macro_f1"],
    "native_model": MODEL_PATH.name,
    "manifest": MANIFEST_PATH.name,
    "metrics": METRICS_PATH.name,
}
print(json.dumps(final_result, indent=2))


## Next steps

1. Download the native JSON model, manifest, and metrics JSON from Kaggle Output.
2. Keep them as a **reference XGBoost experiment**. Do not replace the repository's LOCO-gated six-axis artifact with this model.
3. For the project report, present the Kaggle result alongside the already-run nested-LOCO result and explicitly explain why the two figures are not comparable.
4. Do not attempt to fuse public WASP outputs with public MmCows CUSUM results. They come from different cows and deployments; real fusion needs future same-cow runtime data.
